In [14]:
file_path = "C:/Users/amaca253/Documents/datomatisation/datomatisation-dev/data/demo_data/dogs/breed_traits.csv"

In [15]:

# ------------------------------------------------------------
# Factor Analysis + KMeans clustering for breed_traits.csv
# - Selects #Factors via Parallel Analysis (95th percentile)
# - Fits FactorAnalysis (ML) + varimax rotation
# - Auto-names factors from top loadings
# - Chooses best k (2..8) via Silhouette (tiebreakers: CH↑, DB↓)
# - Names clusters from centroid directions and describes them
# - Writes outputs:
#   * breed_factors_loadings.csv
#   * breed_factor_scores_and_clusters.csv
#   * breed_cluster_summary.csv
# ------------------------------------------------------------
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import FactorAnalysis
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

# -------------------------------
# 1) Load data and select features
# -------------------------------

df = pd.read_csv(file_path)

id_col = "Breed"
non_numeric_cols = ["Coat Type", "Coat Length"]  # exclude categorical text vars
num_cols = [c for c in df.columns if c not in [id_col] + non_numeric_cols]

# Ensure numeric
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

X = df[num_cols].copy().dropna(axis=1, how="all")
X = X.fillna(X.mean(numeric_only=True))

# Standardize so FA operates on correlation structure
scaler = StandardScaler()
Z = scaler.fit_transform(X)

# -------------------------------
# 2) Parallel Analysis (95th pct)
# -------------------------------
def corr_eigenvalues(data_std: np.ndarray) -> np.ndarray:
    R = np.corrcoef(data_std, rowvar=False)
    return np.linalg.eigvalsh(R)[::-1]

obs_eigs = corr_eigenvalues(Z)
rng = np.random.default_rng(42)
n_iter = 500
rand_eigs = np.zeros((n_iter, Z.shape[1]))
for i in range(n_iter):
    R = rng.standard_normal(size=Z.shape)
    R = (R - R.mean(axis=0)) / R.std(axis=0, ddof=1)
    rand_eigs[i, :] = corr_eigenvalues(R)

mean_rand = rand_eigs.mean(axis=0)
p95_rand = np.percentile(rand_eigs, 95, axis=0)

n_factors_mean = int(np.sum(obs_eigs > mean_rand))
n_factors_p95 = int(np.sum(obs_eigs > p95_rand))
best_n_factors = n_factors_p95 if n_factors_p95 >= 1 else max(1, n_factors_mean)

# -------------------------------
# 3) Factor Analysis + varimax
# -------------------------------
fa = FactorAnalysis(n_components=best_n_factors, random_state=42).fit(Z)
loadings = fa.components_.T  # (n_features x n_factors)

def varimax(Phi, gamma=1.0, q=50, tol=1e-6):
    """Orthogonal varimax rotation."""
    p, k = Phi.shape
    R = np.eye(k)
    d = 0.0
    for _ in range(q):
        d_old = d
        Lambda = Phi @ R
        u, s, vh = np.linalg.svd(
            Phi.T @ (Lambda**3 - (gamma/p) * Lambda @ np.diag(np.sum(Lambda**2, axis=0)))
        )
        R = u @ vh
        d = s.sum()
        if d_old != 0 and (d / d_old) < (1 + tol):
            break
    return Phi @ R, R

rot_loadings, R = varimax(loadings)
raw_scores = fa.transform(Z)
rot_scores = raw_scores @ R

feature_names = X.columns.tolist()
factor_names = [f"Factor_{i+1}" for i in range(best_n_factors)]
loadings_df = pd.DataFrame(rot_loadings, index=feature_names, columns=factor_names)

# -------------------------------
# 4) Name factors from top loadings
# -------------------------------
def suggest_factor_name(var_list):
    v = [s.lower() for s in var_list]
    # Sociability & family
    if any(("affectionate" in s) or ("young children" in s) or ("other dogs" in s) for s in v):
        if any(("openness to strangers" in s) for s in v):
            return "Sociability & friendliness"
        return "Family-friendliness"
    # Trainability/adaptability/mental
    if any(("trainability" in s) or ("adaptability" in s) or ("mental stimulation" in s) for s in v):
        return "Trainability & adaptability"
    # Energy/Play/Barking
    if any(("energy level" in s) or ("playfulness" in s) for s in v):
        return "Energy & play drive"
    if any(("barking level" in s) for s in v):
        return "Vocality & alerting"
    # Watchfulness
    if any(("watchdog" in s) or ("protective" in s) for s in v):
        return "Guarding & vigilance"
    # Care/maintenance
    if any(("shedding" in s) or ("grooming" in s) or ("drooling" in s) for s in v):
        return "Coat care & drooling"
    if any(("openness to strangers" in s) for s in v):
        return "Openness to strangers"
    return "General temperament"

proposed = []
for j in range(best_n_factors):
    top_vars = loadings_df.iloc[:, j].abs().sort_values(ascending=False).head(5).index.tolist()
    proposed.append(suggest_factor_name(top_vars))

# Ensure unique names
final_factor_names, seen = [], {}
for nm in proposed:
    if nm not in seen:
        seen[nm] = 1
        final_factor_names.append(nm)
    else:
        seen[nm] += 1
        final_factor_names.append(f"{nm} ({seen[nm]})")

loadings_df.columns = final_factor_names

# -------------------------------
# 5) Cluster on factor scores
# -------------------------------
score_scaler = StandardScaler()
Z_scores = score_scaler.fit_transform(rot_scores)

ks = list(range(2, min(9, len(df))))
res = []
for k in ks:
    km = KMeans(n_clusters=k, n_init=50, random_state=42)
    labels_k = km.fit_predict(Z_scores)
    res.append({
        "k": k,
        "silhouette": silhouette_score(Z_scores, labels_k),
        "calinski_harabasz": calinski_harabasz_score(Z_scores, labels_k),
        "davies_bouldin": davies_bouldin_score(Z_scores, labels_k),
    })
res_df = pd.DataFrame(res)
best = res_df.sort_values(["silhouette", "calinski_harabasz", "davies_bouldin"],
                          ascending=[False, False, True]).iloc[0]
best_k = int(best["k"])

km_final = KMeans(n_clusters=best_k, n_init=100, random_state=42)
labels = km_final.fit_predict(Z_scores)
centroids_z = km_final.cluster_centers_

# -------------------------------
# 6) Name clusters + describe
# -------------------------------
HIGH, LOW = 0.5, -0.5

cluster_names = []
for c in range(best_k):
    z = centroids_z[c]
    order = np.argsort(-np.abs(z))
    top_idxs = order[:min(3, len(final_factor_names))]
    parts = []
    for i in top_idxs:
        sign = "" if z[i] >= 0 else "Low "
        parts.append(sign + final_factor_names[i])
    cluster_names.append(", ".join(parts))

def describe_cluster(z_centroid, factor_names):
    magnitude = np.linalg.norm(z_centroid)
    overview = "This cluster groups breeds with a {} overall profile across the extracted factors.".format(
        "strong" if magnitude > 1.2 else "moderate"
    )
    strengths = []
    weaknesses = []
    for i, nm in enumerate(factor_names):
        if z_centroid[i] >= HIGH:
            strengths.append(nm.lower())
        if z_centroid[i] <= LOW:
            weaknesses.append(nm.lower())
    s2 = "They excel in {}.".format(", ".join(strengths)) if strengths else "Their strengths are balanced without standout peaks."
    s3 = "Potential weaknesses include {}.".format(", ".join(weaknesses)) if weaknesses else "No clear weaknesses emerge relative to the sample."
    return "{} {} {}".format(overview, s2, s3)

descriptions = [describe_cluster(centroids_z[c], final_factor_names) for c in range(best_k)]

# -------------------------------
# 7) Save outputs
# -------------------------------
scores_df = pd.DataFrame(rot_scores, columns=final_factor_names)
scores_df.insert(0, "Breed", df[id_col].values)
scores_df["cluster_id"] = labels
scores_df["cluster_name"] = [cluster_names[c] for c in labels]

loadings_df.to_csv("breed_factors_loadings.csv", index=True)
scores_df.to_csv("breed_factor_scores_and_clusters.csv", index=False)

cluster_summary = []
for c in range(best_k):
    row = {"cluster_id": c, "cluster_name": cluster_names[c], "n_breeds": int((labels == c).sum())}
    for i, nm in enumerate(final_factor_names):
        row["{} (z-mean)".format(nm)] = float(centroids_z[c, i])
    cluster_summary.append(row)
pd.DataFrame(cluster_summary).to_csv("breed_cluster_summary.csv", index=False)

# Optional: print console summary
print("Selected #Factors:", best_n_factors)
print("Factor names:", final_factor_names)
print("Selected k:", best_k)
print(pd.DataFrame(cluster_summary)[["cluster_id", "cluster_name", "n_breeds"]])
for c, d in enumerate(descriptions):
    print("\nCluster {} — {}:\n{}".format(c, cluster_names[c], d))


C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.p

Selected #Factors: 4
Factor names: ['Trainability & adaptability', 'Sociability & friendliness', 'Sociability & friendliness (2)', 'Family-friendliness']
Selected k: 5
   cluster_id                                       cluster_name  n_breeds
0           0  Low Sociability & friendliness, Family-friendl...        37
1           1  Low Sociability & friendliness (2), Trainabili...        40
2           2  Sociability & friendliness (2), Sociability & ...        50
3           3  Low Trainability & adaptability, Sociability &...        41
4           4  Low Family-friendliness, Trainability & adapta...        26

Cluster 0 — Low Sociability & friendliness, Family-friendliness, Trainability & adaptability:
This cluster groups breeds with a strong overall profile across the extracted factors. They excel in family-friendliness. Potential weaknesses include sociability & friendliness.

Cluster 1 — Low Sociability & friendliness (2), Trainability & adaptability, Sociability & friendliness:
Th

Great Pyrenees’ core strengths are clear: they are extremely affectionate with family and highly protective (≈100th percentile on both), with slightly above‑average openness to strangers and other dogs (≈61st and ≈65th percentiles).
They are below average in trainability and adaptability (≈37th and ≈31st percentiles) and maintenance‑heavy, showing very high shedding and drooling (≈91st and ≈95th percentiles), while playfulness/energy are closer to average and barking tends to be somewhat elevated.
Overall, they align with a cluster marked by lower “family‑friendliness” (driven by grooming/drool burden) and lower trainability/adaptability—an affectionate, steadfast guardian best suited to experienced homes.
Fournissez vos commentaires sur BizChat

In [17]:

# ------------------------------------------------------------
# Great Pyrenees: 3-sentence summary from FA + clustering
# Works directly on 'breed_traits.csv'
# ------------------------------------------------------------
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import FactorAnalysis
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

# -------------------------------
# 0) Configuration
# -------------------------------
CSV_PATH = file_path
TARGET_BREED = "Great Pyrenees"  # <-- change to any breed in the file
RANDOM_STATE = 42
N_ITER_PARALLEL = 500
K_MIN, K_MAX = 2, 8
HIGH, LOW = 0.5, -0.5  # thresholds to decide cluster strengths/weaknesses from z-centroids

METRIC_COLS = [
    "Affectionate With Family",
    "Good With Young Children",
    "Good With Other Dogs",
    "Openness To Strangers",
    "Playfulness Level",
    "Watchdog/Protective Nature",
    "Adaptability Level",
    "Trainability Level",
    "Energy Level",
    "Barking Level",
    "Mental Stimulation Needs",
    "Shedding Level",
    "Coat Grooming Frequency",
    "Drooling Level",
]

# -------------------------------
# 1) Utilities
# -------------------------------
def normalize_name(s: str) -> str:
    """Normalize breed names (replace NBSP, trim, lowercase)."""
    return str(s).replace("\u00A0", " ").strip().lower()

def corr_eigenvalues(data_std: np.ndarray) -> np.ndarray:
    """Eigenvalues (descending) of correlation matrix for standardized data."""
    R = np.corrcoef(data_std, rowvar=False)
    return np.linalg.eigvalsh(R)[::-1]

def varimax(Phi: np.ndarray, gamma: float = 1.0, q: int = 50, tol: float = 1e-6):
    """Orthogonal varimax rotation for loadings."""
    p, k = Phi.shape
    R = np.eye(k)
    d = 0.0
    for _ in range(q):
        d_old = d
        Lambda = Phi @ R
        u, s, vh = np.linalg.svd(
            Phi.T @ (Lambda**3 - (gamma / p) * Lambda @ np.diag(np.sum(Lambda**2, axis=0)))
        )
        R = u @ vh
        d = s.sum()
        if d_old != 0 and (d / d_old) < (1 + tol):
            break
    return Phi @ R, R

def suggest_factor_name(top_vars):
    """Rule-based factor naming from top-loading variables."""
    v = [s.lower() for s in top_vars]
    # Sociability & family
    if any(("affectionate" in s) or ("young children" in s) or ("other dogs" in s) for s in v):
        if any(("openness to strangers" in s) for s in v):
            return "Sociability & friendliness"
        return "Family-friendliness"
    # Trainability/adaptability/mental
    if any(("trainability" in s) or ("adaptability" in s) or ("mental stimulation" in s) for s in v):
        return "Trainability & adaptability"
    # Energy/play/barking
    if any(("energy level" in s) or ("playfulness" in s) for s in v):
        return "Energy & play drive"
    if any(("barking level" in s) for s in v):
        return "Vocality & alerting"
    # Guarding
    if any(("watchdog" in s) or ("protective" in s) for s in v):
        return "Guarding & vigilance"
    # Coat/drool/grooming
    if any(("shedding" in s) or ("grooming" in s) or ("drooling" in s) for s in v):
        return "Coat care & drooling"
    if any(("openness to strangers" in s) for s in v):
        return "Openness to strangers"
    return "General temperament"

def name_cluster_from_centroid(z_centroid: np.ndarray, factor_names):
    """Readable cluster name from the top 2–3 absolute factor directions."""
    order = np.argsort(-np.abs(z_centroid))
    top = order[: min(3, len(factor_names))]
    parts = []
    for i in top:
        sign = "" if z_centroid[i] >= 0 else "Low "
        parts.append(sign + factor_names[i])
    return ", ".join(parts)

def percentile_rank(values: np.ndarray, x: float) -> float:
    """Percentile rank of x among values: percentage of values <= x."""
    return float(np.sum(values <= x) / np.sum(~np.isnan(values)) * 100.0)

# -------------------------------
# 2) Load & prepare data
# -------------------------------


if "Breed" not in df.columns:
    raise ValueError("Column 'Breed' not found in the CSV.")

id_col = "Breed"
non_numeric_cols = ["Coat Type", "Coat Length"]  # keep numeric 1–5 scales
numeric_cols = [c for c in df.columns if c not in [id_col] + non_numeric_cols]

# Ensure numeric
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

X = df[numeric_cols].copy().dropna(axis=1, how="all")
X = X.fillna(X.mean(numeric_only=True))

# Standardize so FA uses correlation structure
scaler = StandardScaler()
Z = scaler.fit_transform(X)

# -------------------------------
# 3) Parallel Analysis -> #Factors
# -------------------------------
obs_eigs = corr_eigenvalues(Z)
rng = np.random.default_rng(RANDOM_STATE)
rand_eigs = np.zeros((N_ITER_PARALLEL, Z.shape[1]))
for i in range(N_ITER_PARALLEL):
    S = rng.standard_normal(size=Z.shape)
    S = (S - S.mean(axis=0)) / S.std(axis=0, ddof=1)
    rand_eigs[i, :] = corr_eigenvalues(S)
p95_rand = np.percentile(rand_eigs, 95, axis=0)
n_factors = int(np.sum(obs_eigs > p95_rand))
n_factors = max(1, n_factors)

# -------------------------------
# 4) Factor Analysis + varimax
# -------------------------------
fa = FactorAnalysis(n_components=n_factors, random_state=RANDOM_STATE).fit(Z)
loadings = fa.components_.T  # (features x factors)
rot_loadings, R = varimax(loadings)
raw_scores = fa.transform(Z)
rot_scores = raw_scores @ R

# Factor naming
feature_names = X.columns.tolist()
Ldf = pd.DataFrame(rot_loadings, index=feature_names, columns=[f"Factor_{i+1}" for i in range(n_factors)])

proposed = []
for j in range(n_factors):
    top_vars = Ldf.iloc[:, j].abs().sort_values(ascending=False).head(5).index.tolist()
    proposed.append(suggest_factor_name(top_vars))

# Ensure unique factor names
final_factor_names, seen = [], {}
for nm in proposed:
    if nm not in seen:
        seen[nm] = 1
        final_factor_names.append(nm)
    else:
        seen[nm] += 1
        final_factor_names.append("{} ({})".format(nm, seen[nm]))

# -------------------------------
# 5) Choose k and cluster on factor scores
# -------------------------------
score_scaler = StandardScaler()
Z_scores = score_scaler.fit_transform(rot_scores)

k_candidates = list(range(K_MIN, min(K_MAX + 1, len(df))))
metrics = []
for k in k_candidates:
    km = KMeans(n_clusters=k, n_init=50, random_state=RANDOM_STATE)
    labels_k = km.fit_predict(Z_scores)
    metrics.append({
        "k": k,
        "silhouette": silhouette_score(Z_scores, labels_k),
        "ch": calinski_harabasz_score(Z_scores, labels_k),
        "db": davies_bouldin_score(Z_scores, labels_k),
    })
res_df = pd.DataFrame(metrics)
best = res_df.sort_values(["silhouette", "ch", "db"], ascending=[False, False, True]).iloc[0]
best_k = int(best["k"])

km_final = KMeans(n_clusters=best_k, n_init=100, random_state=RANDOM_STATE)
labels = km_final.fit_predict(Z_scores)
centroids_z = km_final.cluster_centers_

# Cluster names
cluster_names = [name_cluster_from_centroid(centroids_z[c], final_factor_names) for c in range(best_k)]

# -------------------------------
# 6) Build Great Pyrenees 3-sentence summary
# -------------------------------
# Locate the breed (normalize NBSPs, case, spaces)
mask = df["Breed"].apply(normalize_name) == normalize_name(TARGET_BREED)
if not mask.any():
    # Try a fuzzy contains on "Pyren"
    mask = df["Breed"].str.contains("Pyren", case=False, na=False)
    if not mask.any():
        raise ValueError("Breed '{}' not found in the CSV.".format(TARGET_BREED))

row_idx = df.index[mask][0]
breed_name_canonical = df.loc[row_idx, "Breed"]

scores_df = pd.DataFrame(rot_scores, columns=final_factor_names)
scores_df.insert(0, "Breed", df["Breed"].values)
scores_df["cluster_id"] = labels
scores_df["cluster_name"] = scores_df["cluster_id"].apply(lambda c: cluster_names[c])

assign_row = scores_df.iloc[row_idx]
player_cluster_name = assign_row["cluster_name"]

# Percentiles for strengths/weaknesses
percentiles = {}
for col in METRIC_COLS:
    if col in df.columns:
        val = float(df.loc[row_idx, col])
        arr = pd.to_numeric(df[col], errors="coerce").values
        pr = percentile_rank(arr, val)
        percentiles[col] = pr

# Build three sentences (concise)
# S1: strengths -> highlight top clear strengths
strength_bits = []
# Family affection, protection, openness, other dogs
if "Affectionate With Family" in percentiles and percentiles["Affectionate With Family"] >= 90:
    strength_bits.append("extremely affectionate with family (≈{}th percentile)".format(int(round(percentiles["Affectionate With Family"]))))
if "Watchdog/Protective Nature" in percentiles and percentiles["Watchdog/Protective Nature"] >= 90:
    strength_bits.append("highly protective (≈{}th percentile)".format(int(round(percentiles["Watchdog/Protective Nature"]))))
if "Openness To Strangers" in percentiles and percentiles["Openness To Strangers"] >= 60:
    strength_bits.append("somewhat open to strangers (≈{}th percentile)".format(int(round(percentiles["Openness To Strangers"]))))
if "Good With Other Dogs" in percentiles and percentiles["Good With Other Dogs"] >= 60:
    strength_bits.append("slightly above‑average with other dogs (≈{}th percentile)".format(int(round(percentiles["Good With Other Dogs"]))))

if strength_bits:
    s1 = "{}’ core strengths: {}.".format(breed_name_canonical, ", ".join(strength_bits))
else:
    s1 = "{} shows solid strengths without a single standout metric.".format(breed_name_canonical)

# S2: average/weak areas -> trainability/adaptability low; heavy maintenance; moderate energy/playfulness; barking a bit high
weak_bits = []
if "Trainability Level" in percentiles and percentiles["Trainability Level"] <= 40:
    weak_bits.append("below‑average trainability (≈{}th percentile)".format(int(round(percentiles["Trainability Level"]))))
if "Adaptability Level" in percentiles and percentiles["Adaptability Level"] <= 40:
    weak_bits.append("below‑average adaptability (≈{}th percentile)".format(int(round(percentiles["Adaptability Level"]))))
if "Shedding Level" in percentiles and percentiles["Shedding Level"] >= 85:
    weak_bits.append("very high shedding (≈{}th percentile)".format(int(round(percentiles["Shedding Level"]))))
if "Drooling Level" in percentiles and percentiles["Drooling Level"] >= 85:
    weak_bits.append("heavy drooling (≈{}th percentile)".format(int(round(percentiles["Drooling Level"]))))
# contextual notes (not strictly weaknesses)
avg_notes = []
if "Playfulness Level" in percentiles and 40 <= percentiles["Playfulness Level"] <= 60:
    avg_notes.append("playfulness around average")
if "Energy Level" in percentiles and 40 <= percentiles["Energy Level"] <= 60:
    avg_notes.append("energy around average")
if "Barking Level" in percentiles and percentiles["Barking Level"] >= 60:
    avg_notes.append("barking somewhat elevated")

parts = []
if weak_bits:
    parts.append("They are " + ", ".join(weak_bits))
if avg_notes:
    if parts:
        parts.append("while " + ", ".join(avg_notes))
    else:
        parts.append("They show " + ", ".join(avg_notes))
s2 = (parts[0] + ".") if parts else "Across other dimensions, their profile is generally balanced without clear weaknesses."

# S3: concluding statement with cluster name
s3 = "Overall, they align with a '{}' cluster—an affectionate, steadfast guardian best suited to experienced homes.".format(player_cluster_name)

# Print the three-sentence summary
print(s1)
print(s2)
print(s3)



C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\amaca253\AppData\Local\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.p

Great Pyrenees’ core strengths: extremely affectionate with family (≈100th percentile), highly protective (≈100th percentile), somewhat open to strangers (≈61th percentile), slightly above‑average with other dogs (≈65th percentile).
They are below‑average trainability (≈37th percentile), below‑average adaptability (≈31th percentile), very high shedding (≈91th percentile), heavy drooling (≈95th percentile).
Overall, they align with a 'Low Family-friendliness, Trainability & adaptability, Sociability & friendliness (2)' cluster—an affectionate, steadfast guardian best suited to experienced homes.
